# Nosotros contra Charilaou, con todo lo demas fijo

La comparacion de la seccion 0 contra **arXiv:2106.11286** vale poco tal como
esta: su cifra de **1.55-1.63 ns por volteo** esta medida en una **Tesla V100**
con un Hamiltoniano mas liviano que el nuestro, y nuestra cifra de 0.129 ns
esta medida en una **H100** con 5000 replicas de un disco de 5245 sitios.
Varian el modelo, el hardware y el metodo a la vez, asi que el cociente de 12x
no mide ninguno de los tres.

Este cuadernillo los fija de a uno.

| | que se fija | que queda libre |
|---|---|---|
| **T1** | mismo dispositivo, mismo modelo, misma geometria | **el metodo** |
| **T2** | mismo modelo, nuestro metodo | el dispositivo, acotado entre P100 y A100 |
| **T3** | su modelo, nuestro metodo, nuestro dispositivo | contra su cifra publicada |

## Su metodo, que no es checkerboard

El paper **rechaza** la descomposicion en tablero, con el argumento de que
restringe la simetria de la red, y a cambio acepta una carrera de
lectura/escritura que mitiga solo manteniendo `n` suficientemente grande:

> *three linearised 1-D arrays, one per component, in global memory; N_b blocks
> of n spins; one spin per block chosen at random per step; dE computed locally;
> cuRAND Mersenne-Twister*

Nuestro metodo paga un coloreo y a cambio tiene un halo sin carreras y sin
barrera. **T1 es exactamente lo que cuesta o rinde ese intercambio.**

La carrera se **reproduce, no se arregla**: arreglarla mediria algo que el
paper no afirma.

## Tres cosas que juegan en nuestra contra, a proposito

1. **Una sola replica.** Nuestro 0.129 ns sale de 5000 replicas y el piso por
   dispositivo es real. Aqui todos los brazos corren **un solo sistema** en
   **su** geometria: cubica simple 128x128x128 con PBC en los tres ejes.
2. **Su `n` se re-optimiza.** El optimo de `n=8` lo hallo en una V100;
   reutilizarlo en otra GPU lo perjudicaria, asi que se barre `n` y se reporta
   **su mejor**, igual que se barre nuestra altura de bloque.
3. **Actualizacion en sitio.** Su kernel escribe sobre los mismos arreglos. Si
   el anfitrion asignara un buffer nuevo por paso habria que copiar 2.1 millones
   de espines en cada uno y su metodo mediria mucho peor de lo que es. El
   driver **alinea entradas con salidas y se niega a cronometrar** si el
   `jax_triton` instalado no lo permite.

## Nada se cronometra antes de que la fisica pase

`verify_host` comprueba las formulas solo con numpy (corre sin GPU);
`verify_device` compara el kernel contra un motor de referencia en JAX puro con
**la misma aleatoriedad**, donde la unica diferencia admisible es aritmetica y
el conteo de espines distintos tiene que ser **cero**; `verify_theirs` es
estadistico, porque su carrera lo hace no determinista por construccion.


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi",
  "--query-gpu=name,memory.total,clocks.max.sm,driver_version,compute_cap",
  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'jax[cuda12]', 'jax-triton'], check=False)
print('instalado')


In [ ]:
import glob, os, re, subprocess

def ver(p):
    try:
        o = subprocess.run([p, "--version"], capture_output=True,
                           text=True).stdout
        m = re.search(r"release (\d+)\.(\d+)", o)
        return tuple(map(int, m.groups())) if m else (0, 0)
    except Exception:
        return (0, 0)

pats = ["/usr/local/lib/python3*/dist-packages/nvidia/cuda_nvcc/bin/ptxas",
        "/usr/local/lib/python3*/site-packages/nvidia/cuda_nvcc/bin/ptxas",
        "/opt/conda/lib/python3*/site-packages/nvidia/cuda_nvcc/bin/ptxas",
        "/usr/local/lib/python3*/dist-packages/triton/backends/nvidia/bin/ptxas",
        "/opt/conda/lib/python3*/site-packages/triton/backends/nvidia/bin/ptxas",
        "/usr/local/cuda*/bin/ptxas"]
found = sorted({p for q in pats for p in glob.glob(q)})
rows = [(ver(p), p) for p in found]
print("ptxas encontrados:")
for v, p in sorted(rows):
    tag = "   <- el de triton" if "triton" in p else ""
    print("  CUDA %d.%-3d %s%s" % (v[0], v[1], p, tag))

xla = [(v, p) for v, p in rows if "triton" not in p and v > (0, 0)]
if xla:
    # XLA's is the limiting half, so Triton gets pinned to it.
    v, p = min(xla)
    os.environ["TRITON_PTXAS_PATH"] = p
    print("\nTRITON_PTXAS_PATH -> CUDA %d.%d  %s" % (v[0], v[1], p))
else:
    print("\nAVISO: no se hallo un ptxas fuera de triton; se deja como esta.")


In [ ]:
import sys
for m in [k for k in list(sys.modules) if k.split(".")[0] in
          ("jax", "jaxlib", "triton", "jax_triton")]:
    del sys.modules[m]
import jax, jax.numpy as jnp
print("jax", jax.__version__, jax.devices())
try:
    import triton, triton.language as tl, jax_triton as jt
    print("triton", triton.__version__,
          "| jax_triton", getattr(jt, "__version__", "?"))
except Exception as e:
    print("FALTA:", type(e).__name__, e)
    raise SystemExit

# One tiny kernel, so a toolchain failure shows HERE and not buried forty
# minutes into the benchmark.
@triton.jit
def _add1(x_ptr, o_ptr, N: tl.constexpr):
    i = tl.arange(0, N)
    tl.store(o_ptr + i, tl.load(x_ptr + i) + 1.0)

try:
    x = jnp.arange(8, dtype=jnp.float32)
    y = jt.triton_call(x, kernel=_add1, grid=(1,), N=8,
                       out_shape=jax.ShapeDtypeStruct((8,), jnp.float32))
    ok = bool((y == x + 1).all())
    print("\nprueba minima de Triton:", "PASA" if ok else "FALLA", "->", y)
except Exception as e:
    print("\nprueba minima de Triton: FALLA  %s: %s"
          % (type(e).__name__, str(e)[:500]))


In [ ]:
import inspect, jax_triton as jt
sig = inspect.signature(jt.triton_call)
has = "input_output_aliases" in sig.parameters
print("jax_triton.triton_call acepta input_output_aliases:", has)
if not has:
    print("\nSIN ALIAS su metodo no se puede medir con justicia:")
    print("cada paso copiaria los 2.1e6 espines intactos.")
    print("parametros disponibles:", list(sig.parameters))


In [ ]:
# bench_charilaou.py -- byte-identical to the repository copy. sha256 754cf89050a17b05e0052df7e16a3af19035e7cae53c431a6dd7c5e8f9bbeed0
import hashlib
_src = '"""Three controlled comparisons against Charilaou, arXiv:2106.11286.\n\nThe comparison in the thesis section rests on numbers read from that paper,\nmeasured on a Tesla V100 with a Hamiltonian lighter than ours. Three things\ndiffer at once -- the model, the hardware and the method -- so the twelvefold\nraw ratio decomposes into nothing in particular. This file fixes them one at a\ntime:\n\n  T1  same device, same model, THEIR method against OURS\n  T2  our method and their model on devices that bracket the V100\n  T3  our devices running their model, against their reported figure\n\nWhat the paper specifies, and what is reproduced here\n-----------------------------------------------------\nModel    H = -J sum_<ij> S_i.S_j - K sum_i (S_i.e)^2, uniaxial, e = z\n         J = 1, K = 0.01, |S| = 1\nLattice  simple cubic, periodic on all three axes, 128^3 for the headline\nProtocol 1e4 MCS thermalisation + 1e4 MCS averaging at T = 0.9 T_C,\n         with T_C/J = 1.445\nReported 610e6 updates/s = 1.63 ns at 128^3, 630e6 = 1.59 ns at 512^3,\n         1.55 ns as the inset minimum, on a Tesla V100 32 GB\nMethod   three linearised 1-D arrays, one per component, in global memory;\n         N_b blocks of n spins; ONE spin per block chosen at random per step;\n         dE computed locally; cuRAND Mersenne-Twister; NO colouring\n\nThat last line is the point of T1. The paper rejects checkerboard decomposition\nexplicitly, on the grounds that it restricts the lattice symmetry, and accepts\nin exchange a read/write race it mitigates only by keeping n large enough. Our\nmethod pays a colouring and gets a race-free halo. T1 is what that trade costs\nor buys, with everything else held equal.\n\nThree honesty notes, all of which cut against us\n------------------------------------------------\n1. Our headline 0.129 ns is measured at 5000 replicas of a 5245-site disc.\n   His is one system of 2.1e6 spins. The per-device replica floor is real, so\n   comparing those two numbers compares our best parallel configuration against\n   his single-system one. Every arm here runs ONE system at his geometry.\n2. His optimum of n = 8 spins per block was found on a V100. Re-using it on\n   another device would handicap his method, so `theirs` scans n and reports\n   its best, exactly as `ours` scans its block height.\n3. His kernel updates the spin arrays IN PLACE. If the host allocated a fresh\n   output buffer per step, the 2.1e6 unmodified spins would have to be copied\n   every step and his method would measure far worse than it is. The driver\n   therefore aliases its inputs to its outputs and REFUSES to report a time if\n   the installed jax_triton will not do that -- see `_alias_support`.\n\nNothing here is timed until the physics passes. `verify_host` checks the\nformulas with numpy alone, so it runs without a GPU; `verify_device` checks the\nkernel against a pure-JAX reference engine on IDENTICAL randomness, where the\nonly admissible difference is arithmetic and the spin count must be zero.\n`theirs` is nondeterministic by construction -- the race is the method -- so it\nis held to a statistical check instead.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\n\nimport numpy as np\n\ntry:\n    import triton\n    import triton.language as tl\n    HAVE_TRITON = True\nexcept ModuleNotFoundError:                 # host-side checks still run\n    HAVE_TRITON = False\n\n    class _NoTriton:\n        """Lets the module import where Triton is absent, so `verify_host`\n        can check the formulas on a machine with no GPU. Any attempt to build\n        a kernel then fails loudly rather than silently measuring nothing."""\n\n        def jit(self, fn):\n            return fn\n\n        def __getattr__(self, name):\n            raise RuntimeError("Triton is not installed in this environment")\n\n    triton = tl = _NoTriton()\n\n# ----------------------------------------------------------- what he reported\nPAPER = dict(\n    source="arXiv:2106.11286",\n    device="Tesla V100 32GB",\n    ns_per_update={"128^3": 1.63, "512^3": 1.59, "inset_min": 1.55},\n    updates_per_s={"128^3": 610e6, "512^3": 630e6},\n    J=1.0, K=0.01, S=1.0, Tc_over_J=1.445, n_per_block=8,\n    mcs_therm=10_000, mcs_avg=10_000,\n)\n# T = 0.9 T_C with T_C/J = 1.445 and J = 1, so beta = 1/(0.9*1.445).\nBETA = 1.0 / (0.9 * 1.445)\n\n\n# =============================================================== our method\n#\n# The shifted-load checkerboard stencil of `triton_halo`, carrying his\n# Hamiltonian instead of ours and periodic on all three axes instead of two.\n# The index arithmetic is the only thing that changes from production: z wraps\n# here, where production clamps it and gates the wrapped plane to zero.\n\n@triton.jit\ndef _ours_kernel(\n    sp_ptr,          # (3, LZ, N, N)  f32   spins in, component-planar\n    tr_ptr,          # (3, LZ, N, N)  f32   trial spins    (HAS_RNG=False)\n    ru_ptr,          # (LZ, N, N)     f32   uniforms       (HAS_RNG=False)\n    beta_ptr,        # (1,)           f32\n    step_ptr,        # (1,)           i32\n    out_ptr,         # (3, LZ, N, N)  f32   spins out\n    de_ptr,          # (LZ, N, N)     f32   dE out         (WRITE_DE=True)\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    PARITY: tl.constexpr,\n    BY: tl.constexpr,\n    BX: tl.constexpr,\n    JEX: tl.constexpr,\n    KAN: tl.constexpr,\n    SO: tl.constexpr,\n    HAS_RNG: tl.constexpr,\n    WRITE_DE: tl.constexpr,\n):\n    z = tl.program_id(0)\n    ty = tl.program_id(1)\n\n    NN = N * N\n    sc = LZ * NN\n\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tl.arange(0, BX)\n    live = (ys < N)[:, None] & (xs < N)[None, :]\n    yc = tl.where(ys < N, ys, 0)\n    xc = tl.where(xs < N, xs, 0)\n\n    # All three axes wrap. N must be even for the parity colouring to close\n    # across the seam, which the host asserts.\n    ypi = tl.where(yc + 1 >= N, 0, yc + 1)\n    ymi = tl.where(yc - 1 < 0, N - 1, yc - 1)\n    xpi = tl.where(xc + 1 >= N, 0, xc + 1)\n    xmi = tl.where(xc - 1 < 0, N - 1, xc - 1)\n    zp = tl.where(z + 1 >= LZ, 0, z + 1)\n    zm = tl.where(z - 1 < 0, LZ - 1, z - 1)\n\n    o_c = yc[:, None] * N + xc[None, :]\n    o_yp = ypi[:, None] * N + xc[None, :]\n    o_ym = ymi[:, None] * N + xc[None, :]\n    o_xp = yc[:, None] * N + xpi[None, :]\n    o_xm = yc[:, None] * N + xmi[None, :]\n\n    par = (xc[None, :] + yc[:, None] + z) % 2\n    upd = live & (par == PARITY)\n\n    beta = tl.load(beta_ptr)\n    s0 = tl.load(sp_ptr + 0 * sc + z * NN + o_c, mask=live, other=0.0)\n    s1 = tl.load(sp_ptr + 1 * sc + z * NN + o_c, mask=live, other=0.0)\n    s2 = tl.load(sp_ptr + 2 * sc + z * NN + o_c, mask=live, other=0.0)\n\n    if HAS_RNG:\n        # one key, four distinct counters: the acceptance variate must share no\n        # input with the trial direction.\n        sd = tl.load(step_ptr)\n        o4 = (z * NN + o_c).to(tl.int32) * 4\n        g0 = tl.randn(sd, o4 + 0)\n        g1 = tl.randn(sd, o4 + 1)\n        g2 = tl.randn(sd, o4 + 2)\n        nrm = tl.sqrt(g0 * g0 + g1 * g1 + g2 * g2) + 1e-30\n        t0 = g0 / nrm * SO\n        t1 = g1 / nrm * SO\n        t2 = g2 / nrm * SO\n        ru = tl.rand(sd, o4 + 3)\n    else:\n        t0 = tl.load(tr_ptr + 0 * sc + z * NN + o_c, mask=live, other=0.0)\n        t1 = tl.load(tr_ptr + 1 * sc + z * NN + o_c, mask=live, other=0.0)\n        t2 = tl.load(tr_ptr + 2 * sc + z * NN + o_c, mask=live, other=0.0)\n        ru = tl.load(ru_ptr + z * NN + o_c, mask=live, other=0.0)\n\n    d0 = t0 - s0\n    d1 = t1 - s1\n    d2 = t2 - s2\n\n    # ---- the stencil, one component at a time so each component\'s six\n    # neighbour blocks die before the next component\'s are loaded.\n    p = sp_ptr + 0 * sc\n    sv0 = (tl.load(p + z * NN + o_xp, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_xm, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_yp, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_ym, mask=live, other=0.0)\n           + tl.load(p + zp * NN + o_c, mask=live, other=0.0)\n           + tl.load(p + zm * NN + o_c, mask=live, other=0.0))\n    p = sp_ptr + 1 * sc\n    sv1 = (tl.load(p + z * NN + o_xp, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_xm, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_yp, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_ym, mask=live, other=0.0)\n           + tl.load(p + zp * NN + o_c, mask=live, other=0.0)\n           + tl.load(p + zm * NN + o_c, mask=live, other=0.0))\n    p = sp_ptr + 2 * sc\n    sv2 = (tl.load(p + z * NN + o_xp, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_xm, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_yp, mask=live, other=0.0)\n           + tl.load(p + z * NN + o_ym, mask=live, other=0.0)\n           + tl.load(p + zp * NN + o_c, mask=live, other=0.0)\n           + tl.load(p + zm * NN + o_c, mask=live, other=0.0))\n\n    # H = -J sum_<ij> S_i.S_j - K sum_i (S_i.z)^2, so a single-site change is\n    #   dE = -J (t - s).v  -  K (t_z^2 - s_z^2).\n    dE = -JEX * (d0 * sv0 + d1 * sv1 + d2 * sv2) \\\n         - KAN * (t2 * t2 - s2 * s2)\n\n    acc = upd & ((dE <= 0.0) | (ru < tl.exp(-beta * tl.minimum(dE, 80.0))))\n    ob = out_ptr + z * NN + o_c\n    tl.store(ob + 0 * sc, tl.where(acc, t0, s0), mask=live)\n    tl.store(ob + 1 * sc, tl.where(acc, t1, s1), mask=live)\n    tl.store(ob + 2 * sc, tl.where(acc, t2, s2), mask=live)\n    if WRITE_DE:\n        tl.store(de_ptr + z * NN + o_c, dE, mask=live)\n\n\n# ============================================================= their method\n#\n# Faithful to the paper: three linearised component arrays in global memory,\n# N_b blocks each owning n contiguous spins, one of those n drawn at random per\n# step, dE computed from the 1-D index, no colouring and no barrier. The race\n# the paper acknowledges is part of the method and is reproduced, not repaired\n# -- repairing it would measure something the paper does not claim.\n#\n# One spin is updated per block, so the lane count is not ours to choose.\n\n@triton.jit\ndef _theirs_kernel(\n    sx_ptr,          # (NT,) f32   x components, linearised\n    sy_ptr,          # (NT,) f32\n    sz_ptr,          # (NT,) f32\n    beta_ptr,        # (1,)  f32\n    step_ptr,        # (1,)  i32\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    N_PER: tl.constexpr,\n    JEX: tl.constexpr,\n    KAN: tl.constexpr,\n    SO: tl.constexpr,\n):\n    blk = tl.program_id(0)\n    sd = tl.load(step_ptr)\n    NN = N * N\n\n    # one of the block\'s n spins, drawn at random\n    pick = (tl.rand(sd, blk * 5 + 0) * N_PER).to(tl.int32)\n    pick = tl.minimum(pick, N_PER - 1)\n    i = blk * N_PER + pick\n\n    # 1-D index back to (z, y, x) on the simple cubic lattice\n    z = i // NN\n    r = i - z * NN\n    y = r // N\n    x = r - y * N\n\n    xp = tl.where(x + 1 >= N, 0, x + 1)\n    xm = tl.where(x - 1 < 0, N - 1, x - 1)\n    yp = tl.where(y + 1 >= N, 0, y + 1)\n    ym = tl.where(y - 1 < 0, N - 1, y - 1)\n    zp = tl.where(z + 1 >= LZ, 0, z + 1)\n    zm = tl.where(z - 1 < 0, LZ - 1, z - 1)\n\n    nb0 = z * NN + y * N + xp\n    nb1 = z * NN + y * N + xm\n    nb2 = z * NN + yp * N + x\n    nb3 = z * NN + ym * N + x\n    nb4 = zp * NN + y * N + x\n    nb5 = zm * NN + y * N + x\n\n    s0 = tl.load(sx_ptr + i)\n    s1 = tl.load(sy_ptr + i)\n    s2 = tl.load(sz_ptr + i)\n\n    vx = (tl.load(sx_ptr + nb0) + tl.load(sx_ptr + nb1)\n          + tl.load(sx_ptr + nb2) + tl.load(sx_ptr + nb3)\n          + tl.load(sx_ptr + nb4) + tl.load(sx_ptr + nb5))\n    vy = (tl.load(sy_ptr + nb0) + tl.load(sy_ptr + nb1)\n          + tl.load(sy_ptr + nb2) + tl.load(sy_ptr + nb3)\n          + tl.load(sy_ptr + nb4) + tl.load(sy_ptr + nb5))\n    vz = (tl.load(sz_ptr + nb0) + tl.load(sz_ptr + nb1)\n          + tl.load(sz_ptr + nb2) + tl.load(sz_ptr + nb3)\n          + tl.load(sz_ptr + nb4) + tl.load(sz_ptr + nb5))\n\n    g0 = tl.randn(sd, blk * 5 + 1)\n    g1 = tl.randn(sd, blk * 5 + 2)\n    g2 = tl.randn(sd, blk * 5 + 3)\n    nrm = tl.sqrt(g0 * g0 + g1 * g1 + g2 * g2) + 1e-30\n    t0 = g0 / nrm * SO\n    t1 = g1 / nrm * SO\n    t2 = g2 / nrm * SO\n    ru = tl.rand(sd, blk * 5 + 4)\n\n    dE = -JEX * ((t0 - s0) * vx + (t1 - s1) * vy + (t2 - s2) * vz) \\\n         - KAN * (t2 * t2 - s2 * s2)\n    beta = tl.load(beta_ptr)\n    acc = (dE <= 0.0) | (ru < tl.exp(-beta * tl.minimum(dE, 80.0)))\n\n    # The host aliases these three inputs onto the three outputs, so writing\n    # back through them IS the paper\'s in-place update of global memory, and\n    # the untouched n-1 spins of the block are never copied.\n    tl.store(sx_ptr + i, tl.where(acc, t0, s0))\n    tl.store(sy_ptr + i, tl.where(acc, t1, s1))\n    tl.store(sz_ptr + i, tl.where(acc, t2, s2))\n\n\n# ============================================ a pure-JAX reference for ours\n#\n# Runs on CPU or GPU, so the formulas are checkable without a device. This is\n# what `verify_device` holds the kernel to, on identical randomness.\n\ndef ref_engine(N, LZ, J=1.0, K=0.01, So=1.0):\n    import jax.numpy as jnp\n\n    zc, yc, xc = np.indices((LZ, N, N))\n    par = jnp.asarray(((xc + yc + zc) % 2).astype(np.int32))\n\n    def nsum(sp):\n        return (jnp.roll(sp, -1, 3) + jnp.roll(sp, 1, 3)\n                + jnp.roll(sp, -1, 2) + jnp.roll(sp, 1, 2)\n                + jnp.roll(sp, -1, 1) + jnp.roll(sp, 1, 1))\n\n    def dE_of(sp, tr):\n        v = nsum(sp)\n        d = tr - sp\n        return (-J * jnp.sum(d * v, axis=0)\n                - K * (tr[2] ** 2 - sp[2] ** 2))\n\n    def substep(sp, tr, ru, beta, parity):\n        dE = dE_of(sp, tr)\n        acc = (par == parity) & ((dE <= 0) | (ru < jnp.exp(\n            -beta * jnp.minimum(dE, 80.0))))\n        return jnp.where(acc[None], tr, sp), dE\n\n    def energy(sp):\n        # each bond once: only the +x, +y, +z translates\n        e = -J * (jnp.sum(sp * jnp.roll(sp, -1, 3))\n                  + jnp.sum(sp * jnp.roll(sp, -1, 2))\n                  + jnp.sum(sp * jnp.roll(sp, -1, 1)))\n        return e - K * jnp.sum(sp[2] ** 2)\n\n    return dict(substep=substep, dE_of=dE_of, energy=energy, nsum=nsum)\n\n\ndef random_state(key, N, LZ, So=1.0):\n    import jax\n    import jax.numpy as jnp\n    sp = jax.random.normal(key, (3, LZ, N, N), jnp.float32)\n    return sp / (jnp.linalg.norm(sp, axis=0, keepdims=True) + 1e-30) * So\n\n\n# =================================================================== drivers\n\ndef _alias_support():\n    """Does the installed jax_triton alias inputs onto outputs?\n\n    Without it, his in-place update becomes a full-array copy per step and the\n    measurement is meaningless, so this decides whether a time may be reported\n    at all rather than being worked around.\n    """\n    import inspect\n    import jax_triton as jt\n    sig = inspect.signature(jt.triton_call)\n    return "input_output_aliases" in sig.parameters\n\n\ndef build_ours(N, LZ, BY=4, J=1.0, K=0.01, So=1.0, has_rng=True):\n    """One pass of our checkerboard, his Hamiltonian, periodic everywhere."""\n    import jax\n    import jax.numpy as jnp\n    import jax_triton as jt\n\n    assert N % 2 == 0, "the parity colouring needs an even N to close under PBC"\n    BX = 1 << (N - 1).bit_length()\n    n_strip = -(-N // BY)\n    dum3 = jnp.zeros((1,), jnp.float32)\n\n    def substep(sp, beta, step, parity, tr=None, ru=None, want_de=False):\n        shape = jax.ShapeDtypeStruct(sp.shape, jnp.float32)\n        de_shape = jax.ShapeDtypeStruct(\n            (LZ, N, N) if want_de else (1,), jnp.float32)\n        out, de = jt.triton_call(\n            sp,\n            dum3 if tr is None else tr,\n            dum3 if ru is None else ru,\n            jnp.float32(beta).reshape(1), jnp.int32(step).reshape(1),\n            kernel=_ours_kernel, out_shape=(shape, de_shape),\n            grid=(LZ, n_strip), N=N, LZ=LZ, PARITY=int(parity), BY=BY, BX=BX,\n            JEX=float(J), KAN=float(K), SO=float(So),\n            HAS_RNG=bool(tr is None), WRITE_DE=bool(want_de), num_warps=4,\n        )\n        return (out, de) if want_de else out\n\n    def sweep(sp, beta, step):\n        sp = substep(sp, beta, step * 2 + 0, 0)\n        return substep(sp, beta, step * 2 + 1, 1)\n\n    return dict(substep=substep, sweep=sweep, BY=BY, BX=BX)\n\n\ndef build_theirs(N, LZ, n_per=8, J=1.0, K=0.01, So=1.0):\n    """One of his steps: N_b blocks, one random spin updated in each."""\n    import jax\n    import jax.numpy as jnp\n    import jax_triton as jt\n\n    if not _alias_support():\n        raise RuntimeError(\n            "this jax_triton cannot alias inputs onto outputs, so his in-place "\n            "update would become a full-array copy per step; refusing to time it")\n\n    NT = LZ * N * N\n    assert NT % n_per == 0, "n must divide the site count"\n    n_blocks = NT // n_per\n    shp = jax.ShapeDtypeStruct((NT,), jnp.float32)\n\n    def step_fn(sx, sy, sz, beta, step):\n        return jt.triton_call(\n            sx, sy, sz,\n            jnp.float32(beta).reshape(1), jnp.int32(step).reshape(1),\n            kernel=_theirs_kernel, out_shape=(shp, shp, shp),\n            grid=(n_blocks,), N=N, LZ=LZ, N_PER=int(n_per),\n            JEX=float(J), KAN=float(K), SO=float(So), num_warps=1,\n            input_output_aliases={0: 0, 1: 1, 2: 2},\n        )\n\n    return dict(step=step_fn, n_blocks=n_blocks, n_per=n_per,\n                updates_per_step=n_blocks)\n\n\n# ============================================================== verification\n\ndef verify_host(N=8, LZ=8, J=1.0, K=0.01, So=1.0, tol=2e-6):\n    """The formulas, with numpy alone. Runs without a GPU."""\n    import jax\n    import jax.numpy as jnp\n\n    rep = []\n    ref = ref_engine(N, LZ, J=J, K=K, So=So)\n    sp = random_state(jax.random.PRNGKey(0), N, LZ, So)\n    spn = np.asarray(sp)\n\n    # 1. the neighbour sum, against an explicit six-term loop with PBC\n    v = np.asarray(ref["nsum"](sp))\n    man = np.zeros_like(spn)\n    for ax, sh in ((3, -1), (3, 1), (2, -1), (2, 1), (1, -1), (1, 1)):\n        man += np.roll(spn, sh, axis=ax)\n    e1 = float(np.abs(v - man).max())\n    rep.append(("neighbour sum vs an explicit PBC loop", e1 < tol, e1))\n\n    # 2. the energy, against a direct bond enumeration over the whole lattice\n    E_ref = float(ref["energy"](sp))\n    E_dir = 0.0\n    for z in range(LZ):\n        for y in range(N):\n            for x in range(N):\n                s = spn[:, z, y, x]\n                for dz, dy, dx in ((0, 0, 1), (0, 1, 0), (1, 0, 0)):\n                    o = spn[:, (z + dz) % LZ, (y + dy) % N, (x + dx) % N]\n                    E_dir += -J * float(s @ o)\n                E_dir += -K * float(s[2] ** 2)\n    e2 = abs(E_ref - E_dir) / max(abs(E_dir), 1e-30)\n    rep.append(("energy vs a direct bond enumeration", e2 < 1e-5, e2))\n\n    # 3. dE, against recomputing the total energy before and after one flip\n    tr = random_state(jax.random.PRNGKey(7), N, LZ, So)\n    dE = np.asarray(ref["dE_of"](sp, tr))\n    z, y, x = LZ // 2, N // 2, N // 3\n    sp2 = spn.copy()\n    sp2[:, z, y, x] = np.asarray(tr)[:, z, y, x]\n    dE_tot = float(ref["energy"](jnp.asarray(sp2))) - E_ref\n    e3 = abs(dE[z, y, x] - dE_tot) / max(abs(dE_tot), 1e-30)\n    rep.append(("dE vs the total-energy difference of one flip", e3 < 1e-4, e3))\n\n    # 4. the colouring closes under PBC, which needs an even N\n    zc, yc, xc = np.indices((LZ, N, N))\n    par = (xc + yc + zc) % 2\n    bad = 0\n    for dz, dy, dx in ((0, 0, 1), (0, 1, 0), (1, 0, 0)):\n        nb = par[(zc + dz) % LZ, (yc + dy) % N, (xc + dx) % N]\n        bad += int((nb == par).sum())\n    rep.append(("every first neighbour is of the opposite colour",\n                bad == 0, float(bad)))\n\n    ok = all(r[1] for r in rep)\n    for name, good, val in rep:\n        print(f"  [{\'ok \' if good else \'FAIL\'}] {name}: {val:.3e}")\n    return ok, rep\n\n\ndef verify_device(N=16, LZ=16, J=1.0, K=0.01, So=1.0):\n    """The kernel against the reference, on IDENTICAL randomness.\n\n    With the randomness held fixed the only admissible difference is\n    arithmetic, so the differing-spin count must be zero: a non-zero count\n    identifies a term transcribed wrongly, not a sampling discrepancy.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    rep = []\n    ref = ref_engine(N, LZ, J=J, K=K, So=So)\n    eng = build_ours(N, LZ, BY=4, J=J, K=K, So=So)\n    sp = random_state(jax.random.PRNGKey(3), N, LZ, So)\n    tr = random_state(jax.random.PRNGKey(4), N, LZ, So)\n    ru = jax.random.uniform(jax.random.PRNGKey(5), (LZ, N, N), jnp.float32)\n\n    for parity in (0, 1):\n        got, de = eng["substep"](sp, BETA, 0, parity, tr=tr, ru=ru,\n                                 want_de=True)\n        want, dE_ref = ref["substep"](sp, tr, ru, BETA, parity)\n        nd = int(np.sum(np.abs(np.asarray(got) - np.asarray(want)) > 1e-6))\n        rep.append((f"one pass, parity {parity}, identical randomness",\n                    nd == 0, float(nd)))\n        ed = float(np.abs(np.asarray(de) - np.asarray(dE_ref)).max())\n        rep.append((f"dE, parity {parity}", ed < 2e-5, ed))\n\n    # the in-kernel generator must put the trial direction on the sphere\n    got = eng["substep"](sp, BETA, 11, 0)\n    nrm = np.linalg.norm(np.asarray(got), axis=0)\n    rep.append(("|s| = So after an in-kernel pass",\n                float(np.abs(nrm - So).max()) < 2e-5,\n                float(np.abs(nrm - So).max())))\n\n    ok = all(r[1] for r in rep)\n    for name, good, val in rep:\n        print(f"  [{\'ok \' if good else \'FAIL\'}] {name}: {val:.3e}")\n    return ok, rep\n\n\ndef verify_theirs(N=16, LZ=16, steps=400, J=1.0, K=0.01, So=1.0):\n    """His method can only be checked statistically: the race is the method.\n\n    Two things must still hold. The spins must stay on the sphere, and the\n    energy must fall from a random start towards the reference engine\'s\n    equilibrium -- a method that accepted everything or nothing would fail both.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    rep = []\n    ref = ref_engine(N, LZ, J=J, K=K, So=So)\n    eng = build_theirs(N, LZ, n_per=8, J=J, K=K, So=So)\n    s = np.asarray(random_state(jax.random.PRNGKey(9), N, LZ, So))\n    E0 = float(ref["energy"](jnp.asarray(s)))\n    sx, sy, sz = (jnp.asarray(s[0].ravel()), jnp.asarray(s[1].ravel()),\n                  jnp.asarray(s[2].ravel()))\n    for t in range(steps):\n        sx, sy, sz = eng["step"](sx, sy, sz, BETA, t)\n    out = np.stack([np.asarray(sx).reshape(LZ, N, N),\n                    np.asarray(sy).reshape(LZ, N, N),\n                    np.asarray(sz).reshape(LZ, N, N)])\n    nrm = np.linalg.norm(out, axis=0)\n    rep.append(("|s| = So throughout", float(np.abs(nrm - So).max()) < 2e-5,\n                float(np.abs(nrm - So).max())))\n    E1 = float(ref["energy"](jnp.asarray(out)))\n    rep.append((f"energy falls from a random start ({E0:.1f} -> {E1:.1f})",\n                E1 < E0, (E1 - E0) / abs(E0)))\n    ok = all(r[1] for r in rep)\n    for name, good, val in rep:\n        print(f"  [{\'ok \' if good else \'FAIL\'}] {name}: {val:.3e}")\n    return ok, rep\n\n\n# ================================================================== timing\n\ndef _sync(x):\n    import jax\n    jax.block_until_ready(x)\n    return x\n\n\ndef time_ours(N=128, LZ=128, sweeps=200, BY=4, J=1.0, K=0.01, So=1.0):\n    """ns per attempted single-site update, one system, his geometry."""\n    import jax\n    eng = build_ours(N, LZ, BY=BY, J=J, K=K, So=So)\n    sp = random_state(jax.random.PRNGKey(1), N, LZ, So)\n\n    def go(s, n):\n        for t in range(n):\n            s = eng["sweep"](s, BETA, t)\n        return s\n\n    _sync(go(sp, 3))                        # compile and warm\n    t0 = time.perf_counter()\n    out = _sync(go(sp, sweeps))\n    dt = time.perf_counter() - t0\n    sites = LZ * N * N\n    return dict(seconds=dt, sweeps=sweeps, sites=sites, BY=BY,\n                ns_per_update=dt / (sweeps * sites) * 1e9,\n                updates_per_s=sweeps * sites / dt,\n                finite=bool(np.isfinite(np.asarray(out)).all()))\n\n\ndef time_theirs(N=128, LZ=128, steps=2000, n_per=8, J=1.0, K=0.01, So=1.0):\n    """The same unit for his method.\n\n    One of his steps attempts N_b = sites/n updates, so the denominator is\n    steps * N_b and not steps * sites.\n    """\n    import jax\n    import jax.numpy as jnp\n    eng = build_theirs(N, LZ, n_per=n_per, J=J, K=K, So=So)\n    NT = LZ * N * N\n    s = np.asarray(random_state(jax.random.PRNGKey(2), N, LZ, So))\n    mk = lambda i: jnp.asarray(s[i].ravel())\n\n    def go(n):\n        a, b, c = mk(0), mk(1), mk(2)\n        for t in range(n):\n            a, b, c = eng["step"](a, b, c, BETA, t)\n        return a, b, c\n\n    _sync(go(3))\n    t0 = time.perf_counter()\n    out = _sync(go(steps))\n    dt = time.perf_counter() - t0\n    attempts = steps * eng["updates_per_step"]\n    return dict(seconds=dt, steps=steps, n_per=n_per, sites=NT,\n                attempts=attempts,\n                ns_per_update=dt / attempts * 1e9,\n                updates_per_s=attempts / dt,\n                finite=bool(np.isfinite(np.asarray(out[0])).all()))\n\n\ndef run(N=128, LZ=128, out=None, label="", sweeps=200, steps=2000,\n        by_scan=(2, 4, 8), n_scan=(1, 2, 4, 8, 16, 32)):\n    """Both methods at his geometry, each scanned for its own best shape."""\n    import jax\n\n    dev = jax.devices()[0]\n    print(f"=== bench_charilaou  {N}x{N}x{LZ}  ({N*N*LZ:,} spins) ===")\n    print(f"    device: {dev.device_kind}   jax {jax.__version__}")\n    print("--- verification; nothing is timed until every check passes")\n    ok_h, rep_h = verify_host()\n    ok_d, rep_d = verify_device()\n    ok_t, rep_t = verify_theirs()\n    if not (ok_h and ok_d and ok_t):\n        raise SystemExit("verification FAILED -- refusing to report a time")\n\n    res = dict(label=label, N=N, LZ=LZ, sites=N * N * LZ, paper=PAPER,\n               device=dev.device_kind, jax=jax.__version__, beta=BETA,\n               verification=dict(\n                   host=[(n, bool(g), float(v)) for n, g, v in rep_h],\n                   ours=[(n, bool(g), float(v)) for n, g, v in rep_d],\n                   theirs=[(n, bool(g), float(v)) for n, g, v in rep_t]))\n\n    print("--- ours (checkerboard, shifted loads), scanning the block height")\n    res["ours"] = []\n    for by in by_scan:\n        try:\n            r = time_ours(N, LZ, sweeps=sweeps, BY=by)\n        except Exception as e:\n            print(f"    BY={by:<3} refused: {type(e).__name__}: {e}")\n            continue\n        res["ours"].append(r)\n        print(f"    BY={by:<3} {r[\'ns_per_update\']:8.4f} ns/update"\n              f"   {r[\'updates_per_s\']/1e6:9.1f} e6 upd/s")\n\n    print("--- theirs (1 of n per block, no colouring), scanning n")\n    res["theirs"] = []\n    for n in n_scan:\n        if (N * N * LZ) % n:\n            continue\n        try:\n            r = time_theirs(N, LZ, steps=steps, n_per=n)\n        except Exception as e:\n            print(f"    n={n:<3} refused: {type(e).__name__}: {e}")\n            continue\n        res["theirs"].append(r)\n        print(f"    n={n:<3}  {r[\'ns_per_update\']:8.4f} ns/update"\n              f"   {r[\'updates_per_s\']/1e6:9.1f} e6 upd/s")\n\n    bo = min((r["ns_per_update"] for r in res["ours"]), default=None)\n    bt = min((r["ns_per_update"] for r in res["theirs"]), default=None)\n    res["best_ours_ns"], res["best_theirs_ns"] = bo, bt\n    res["ratio_theirs_over_ours"] = (bt / bo) if (bo and bt) else None\n    if bo and bt:\n        print(f"--- best ours {bo:.4f} ns | best theirs {bt:.4f} ns"\n              f" | theirs/ours {bt/bo:.2f}x")\n    print(f"    paper on a V100: {PAPER[\'ns_per_update\'][\'128^3\']} ns at"\n          f" 128^3, {PAPER[\'ns_per_update\'][\'inset_min\']} ns minimum")\n\n    if out:\n        os.makedirs(os.path.dirname(out) or ".", exist_ok=True)\n        json.dump(res, open(out, "w"), indent=1)\n        print(f"    wrote {out}")\n    return res\n\n\nif __name__ == "__main__":\n    import argparse\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--N", type=int, default=128)\n    ap.add_argument("--LZ", type=int, default=128)\n    ap.add_argument("--sweeps", type=int, default=200)\n    ap.add_argument("--steps", type=int, default=2000)\n    ap.add_argument("--out", default="")\n    ap.add_argument("--label", default="")\n    ap.add_argument("--host-only", action="store_true",\n                    help="run only the checks that need no GPU")\n    a = ap.parse_args()\n    if a.host_only:\n        ok, _ = verify_host()\n        raise SystemExit(0 if ok else 1)\n    run(a.N, a.LZ, out=a.out or None, label=a.label,\n        sweeps=a.sweeps, steps=a.steps)\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "754cf89050a17b05e0052df7e16a3af19035e7cae53c431a6dd7c5e8f9bbeed0", "bench_charilaou.py alterado"
open("bench_charilaou.py", "w").write(_src)
print("bench_charilaou.py  {} bytes  sha256 ok".format(len(_src)))


In [ ]:
LABEL = 'charilaou_t4'
SWEEPS = 120
STEPS = 1200
BEST_BY = 4
print(LABEL, SWEEPS, STEPS)


## T1, T2 y T3 en este dispositivo

Primero la verificacion; si algo falla, el script aborta y no reporta ningun tiempo.

In [ ]:
import os, json
os.environ.setdefault("TRITON_PTXAS_PATH", os.environ.get("TRITON_PTXAS_PATH", ""))
import bench_charilaou as BC

res = BC.run(N=128, LZ=128, sweeps=SWEEPS, steps=STEPS,
             label=LABEL, out="/kaggle/working/" + LABEL + ".json")
print(json.dumps({k: res[k] for k in
                  ("device", "best_ours_ns", "best_theirs_ns",
                   "ratio_theirs_over_ours")}, indent=1))


## El barrido de tamanos, porque su cifra depende de la red

In [ ]:
# T3 wants his two reported sizes, not just one, because his own figure
# falls from 1.63 ns at 128^3 to 1.59 at 512^3 -- the rate depends on the
# lattice and a single point would not be comparable. 512^3 needs 1.6 GB for
# the three arrays, so it is attempted and allowed to fail on a small card.
import json, bench_charilaou as BC

sizes = []
for n in (64, 128, 192, 256):
    try:
        r = BC.time_ours(n, n, sweeps=max(20, 2_000_000 // (n * n * n // 50)),
                         BY=BEST_BY)
        r["N"] = n
        sizes.append(r)
        print("  %4d^3  %8.4f ns/update   %9.1f e6 upd/s"
              % (n, r["ns_per_update"], r["updates_per_s"] / 1e6))
    except Exception as e:
        print("  %4d^3  rechazado: %s: %s" % (n, type(e).__name__, str(e)[:120]))
json.dump(sizes, open("/kaggle/working/" + LABEL + "_sizes.json", "w"), indent=1)


## Como leer el resultado

- **`theirs/ours` en el mismo dispositivo** es T1, y es el unico numero de este
  cuadernillo del que el hardware ya no es excusa para ninguna de las dos
  partes. Si sale por debajo de 1, su metodo es mas rapido que el nuestro con
  el mismo modelo y la misma GPU, y eso habria que decirlo en la seccion.
- **`best_ours_ns` contra sus 1.55-1.63 ns** es T2/T3. La T4 esta *por debajo*
  de una V100 y la P100 mas todavia, asi que si nuestro numero en T4 ya le gana
  a su numero en V100, la ventaja no puede ser del hardware.
- El barrido de tamanos existe porque **su propia cifra depende de la red**
  (1.63 ns en 128^3, 1.59 en 512^3). Comparar un solo punto contra otro solo
  punto no seria comparar.

Los `.json` quedan en `/kaggle/working/` y se bajan al repositorio.
